# 1. Import the necessary libraries

In [ ]:
# Adds the parent directory to sys.path
import sys

sys.path.append('../')

import statistics
import timeit
from typing import Any

import pandas as pd
import pygmo as pg
from problems.benchmark_problems import get_problem
from pymoo.algorithms.moo.nsga2 import NSGA2
from pymoo.core.problem import Problem
from pymoo.operators.crossover.sbx import SBX
from pymoo.operators.mutation.pm import PM
from pymoo.optimize import minimize
from runners import get_tuned_parameters

from amesh.core import AMESH, AMESHParameters


# 2. Experiment Configuration

In [ ]:
#################### CUSTOMIZABLE ####################

# Objective function configuration (function name, number of objectives, number of decision variables)
experiments = [
    ('zdt1', 2, 10, None), ('zdt2', 2, 10, None), ('zdt3', 2, 10, None), ('zdt4', 2, 10, None), ('zdt6', 2, 10, None),
    ('dtlz1', 3, 10, None), ('dtlz2', 3, 10, None), ('dtlz3', 3, 10, None), ('dtlz4', 3, 10, None), ('dtlz5', 3, 10, None), ('dtlz6', 3, 10, None), ('dtlz7', 3, 10, None),
    ('wfg1', 3, 10, 6), ('wfg2', 3, 10, 6), ('wfg3', 3, 10, 6), ('wfg4', 3, 10, 6), ('wfg5', 3, 10, 6), ('wfg6', 3, 10, 6), ('wfg7', 3, 10, 6), ('wfg8', 3, 10, 6),
    ('wfg9', 3, 10, 6)]

# Execution configuration
num_repeats = 30 # Number of runs for collecting statistics
max_fitness_eval = 15000 # Maximum fitness evaluations (not used if it is None)
population_size = 100 # Population size
random_state = None # Set a seed for random numbers (not used if it is None)

tuning_folder = '../hyperparams' # Folder to get the tuned parameters
######################################################

df_columns = ['Function', 'Algorithm', 'Min Time (s)', 'Max Time (s)', 'Mean Time (s)', 'Std Dev Time (s)', 'Median Time (s)']

# 3. Auxiliar Functions

In [ ]:
# Function to get the fitness function configuration
def get_exp_problem(func_name, objective_dim, position_dim, wfg_k):
	func, position_min_value, position_max_value = get_problem(func_name, n_var=position_dim, n_obj=objective_dim, wfg_k=wfg_k)
	return func, objective_dim, position_dim, position_min_value, position_max_value

def run_amesh(experiment: dict[str, Any],
			 problem: dict[str, Any],
			 parameters: dict[str, Any]) -> None:
	# Get the experiment configuration
	experiment_name = experiment['name']
	# results_folder = experiment['results_folder']
	fine_tuning_folder = experiment['fine_tuning_folder']
	# num_runs = experiment['num_runs']
	max_fitness_eval = experiment['max_fitness_eval']
	population_size = experiment['population_size']
	random_state = experiment['random_state']
    # Get the problem configuration
	fitness = problem['fitness']
	objective_dim = problem['objective_dim']
	decision_dim = problem['decision_dim']
	lower_bound_array = problem['lower_bound_array']
	upper_bound_array = problem['upper_bound_array']

	# Get tunable parameters (check if the parameters were tuned)
	tuned_parameters = get_tuned_parameters(experiment_name, fine_tuning_folder)
	global_best_attribution_type = tuned_parameters['global_best_attribution_type'] if ('global_best_attribution_type' in tuned_parameters) else parameters['global_best_attribution_type']
	dm_pool_type = tuned_parameters['differential_mutation_pool_type'] if ('differential_mutation_pool_type' in tuned_parameters) else parameters['differential_mutation_pool_type']
	dm_operation_type = tuned_parameters['differential_mutation_type'] if ('differential_mutation_type' in tuned_parameters) else parameters['differential_mutation_type']
	personal_guide_array_size = tuned_parameters['personal_guide_array_size'] if ('personal_guide_array_size' in tuned_parameters) else parameters['personal_guide_array_size']

	params = AMESHParameters(objective_dim = objective_dim,
							decision_dim = decision_dim,
							decision_lower_bounds = lower_bound_array,
							decision_upper_bounds = upper_bound_array, 
							population_size = population_size,
							global_guide_method = global_best_attribution_type,
							dm_pool_type = dm_pool_type,
							dm_operation_type = dm_operation_type,
							max_gen = None,
							max_fit_eval = max_fitness_eval,
							max_personal_guides = personal_guide_array_size,
							random_state = random_state)
	mesh = AMESH(params = params, fitness_function = fitness)
	mesh.run()

def run_nsga2(experiment: dict[str, Any],
			  problem: dict[str, Any],
			  parameters: dict[str, Any]) -> None:
	# Get the experiment configuration
	experiment_name = experiment['name']
	# results_folder = experiment['results_folder']
	fine_tuning_folder = experiment['fine_tuning_folder']
	# num_runs = experiment['num_runs']
	max_fitness_eval = experiment['max_fitness_eval']
	population_size = experiment['population_size']
	random_state = experiment['random_state']
    # Get the problem configuration
	fitness_function = problem['fitness']
	objective_dim = problem['objective_dim']
	# decision_dim = problem['decision_dim']
	lower_bound_array = problem['lower_bound_array']
	upper_bound_array = problem['upper_bound_array']
	class PygmoProblem:
		def fitness(self, x):
			return fitness_function(x)
		def get_bounds(self):
			return (lower_bound_array, upper_bound_array)
		def get_nobj(self):
			return objective_dim
	pygmo_problem = PygmoProblem()

	# Get tunable parameters (check if the parameters were tuned)
	tuned_parameters_dict = get_tuned_parameters(experiment_name, fine_tuning_folder)
	recombination_probability = tuned_parameters_dict['recombination_probability'] if ('recombination_probability' in tuned_parameters_dict) else parameters['recombination_probability']
	eta_recombination = tuned_parameters_dict['eta_recombination'] if ('eta_recombination' in tuned_parameters_dict) else parameters['eta_recombination']
	mutation_probability = tuned_parameters_dict['mutation_probability'] if ('mutation_probability' in tuned_parameters_dict) else parameters['mutation_probability']
	eta_mutation = tuned_parameters_dict['eta_mutation'] if ('eta_mutation' in tuned_parameters_dict) else parameters['eta_mutation']

	# Instantiate NSGA2
	crossover = SBX(prob=recombination_probability, prob_var=1.0, eta=eta_recombination)
	mutation = PM(prob=mutation_probability, eta=eta_mutation)
	nsga2 = NSGA2(pop_size=population_size,
				  crossover=crossover,
				  mutation=mutation,
			      eliminate_duplicates=True)

	# Execute NSGA2
	minimize(pygmo_problem, nsga2, ('n_eval', max_fitness_eval), seed=random_state, verbose=False)

# Function to capture errors during parallel execution without stopping the other processes
def safe_run(run_function, params):
	try:
		run_function(*params)
	except Exception as e:
		print(f'Error: {str(e)}')

# 4. Benchmark

## 4.1 A-MESH

In [ ]:
#################### CUSTOMIZABLE ####################
# A-MESH fixed parameters
amesh_memory_size = population_size # Maximum number of particles in memory
config_list = [(0,1,2), (1,1,1), (1,0,0), (0,0,0), (0,2,1),
               (1,1,0), (0,1,0), (0,0,1), (1,2,0), (1,2,3), (0,1,0), (1,0,0),
               (1,2,0), (1,0,0), (0,2,0), (1,2,2), (1,1,3), (1,1,0), (0,2,2), (1,0,2), (0,0,0)]


# A-MESH tunable parameters
# OBS: The function "run_amesh" and "run_amesh_old" will select automatically the tuned parameters if they exists ("hyperparams" folder generated by "fine_tuning.ipynb")
amesh_communication_probability = 0.8 # Communication probability
amesh_mutation_rate = 0.9 # Mutation rate
amesh_personal_guide_array_size = 1 # Number of personal guides
######################################################

info_list = [(exp[0].upper(), f'G{config_list[i][0]+1}S{config_list[i][1]+1}D{config_list[i][2]+1}')
			 for i, exp in enumerate(experiments)
]

In [ ]:
# Set the list of parameters
params_list = [
  	[(F'AMESH_G{config_list[i][0]+1}S{config_list[i][1]+1}D{config_list[i][2]+1}_{exp[0]}_{exp[1]}_{exp[2]}', tuning_folder, max_fitness_eval, population_size, random_state),
    get_exp_problem(exp[0], exp[1], exp[2], exp[3]),
    (amesh_memory_size, *config_list[i]),
    (amesh_communication_probability, amesh_mutation_rate, amesh_personal_guide_array_size)]
    for i, exp in enumerate(experiments)
]

# Collect the times for each experiment
amesh_times = [
    timeit.repeat(
        lambda: safe_run(run_amesh, params),
        repeat=num_repeats,
        number=1
    )
    for params in params_list
]

# Create a DataFrame to store the results
amesh_results = [
  [info_list[i][0],
   'A-MESH ' + info_list[i][1],
   min(t),
   max(t),
   statistics.mean(t),
   statistics.stdev(t) if len(t) > 1 else 0.0,
   statistics.median(t)] for i, t in enumerate(amesh_times)]

amesh_df = pd.DataFrame(amesh_results, columns=df_columns)
amesh_df

  0%|          | 0/15000 [00:00<?, ?it/s]                

In [ ]:
print(amesh_df.to_latex(index=False, float_format="%.3f", escape=False, column_format='lcccccc', label='tab:hv', caption='Hypervolume indicator of the algorithms on the test problems.'))

## 4.2 NSGA-II

In [ ]:
#################### CUSTOMIZABLE ####################

# NSGA2 tunable parameters
# OBS: The function "run_nsga2" will select automatically the tuned parameters if they exists ("hyperparams" folder generated by "fine_tuning.ipynb")
nsga2_recombination_probability = 0.45
nsga2_eta_recombination = 19
nsga2_mutation_probability = 0.06
nsga2_eta_mutation = 14.7
######################################################

# Set the list of parameters
params_list = [
  	[(F'NSGA2_{exp[0]}_{exp[1]}_{exp[2]}', tuning_folder, max_fitness_eval, population_size, random_state),
	 get_exp_problem(exp[0], exp[1], exp[2], exp[3]),
	 (),
	 (nsga2_recombination_probability, nsga2_eta_recombination, nsga2_mutation_probability, nsga2_eta_mutation)]
     for exp in experiments
]

# Collect the times for each experiment
nsga2_times = [
    timeit.repeat(
        lambda: safe_run(run_nsga2, params),
        repeat=num_repeats,
        number=1
    )
    for params in params_list
]

# Create a DataFrame to store the results
nsga2_results = [
  [experiments[i][0].upper(),
   'NSGA2',
   min(t),
   max(t),
   statistics.mean(t),
   statistics.stdev(t) if len(t) > 1 else 0.0,
   statistics.median(t)] for i, t in enumerate(nsga2_times)]

nsga2_df = pd.DataFrame(nsga2_results, columns=df_columns)
nsga2_df

In [ ]:
print(nsga2_df.to_latex(index=False, float_format="%.3f", escape=False, column_format='lcccccc', label='tab:hv', caption='Hypervolume indicator of the algorithms on the test problems.'))